# Build `single_steps_flybody.pkl` from a curated selection

Two-step workflow for producing the per-leg single-step asset consumed by `FlybodyPreprogrammedSteps`:

1. **Replay & select.** Replay the bundled `ball_flybody_clip.npz` on a tethered Flybody (no ground, body welded so claw motion reflects only the commanded joint angles). Render a ventral-view video, plot each claw's body-frame anteroposterior coordinate vs. time with auto-detected AEP markers overlaid, and hand-pick **one step per leg position (F/M/H)** — either side. Save the picks to a JSON file.
2. **Build.** From the recording + the saved selection, resample each picked cycle onto a fixed phase grid, compute the swing fraction as the fraction of timesteps where the body-frame anteroposterior velocity is negative (`np.diff(claw_ap) < 0`, i.e. the claw retreats from AEP toward PEP), and mirror the picked side onto the opposite side (roll/yaw sign flip since the clip is in SeqIKPy / global convention). Write the result, plus full provenance, to `single_steps_flybody.pkl`.

See [`flybody_step_extraction.py`](flybody_step_extraction.py) for the library functions used below.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from flygym_demo.complex_terrain.flybody_step_extraction import (
    LEG_POSITIONS,
    LEGS,
    build_asset_from_selection,
    find_candidate_peps,
    load_replay_recording,
    replay_clip,
    save_asset,
    save_replay_recording,
    save_selection,
)

REPO_ROOT = Path("..").resolve()
CLIP_PATH = REPO_ROOT / "ball_flybody_data/assets/ball_flybody_clip.npz"
ASSET_DIR = REPO_ROOT / "complex_terrain/assets"
RECORDING_PATH = ASSET_DIR / "flybody_replay_recording.pkl"
SELECTION_PATH = ASSET_DIR / "flybody_step_selection.json"
ASSET_PATH = ASSET_DIR / "single_steps_flybody.pkl"
VIDEO_PATH = ASSET_DIR / "ventral_replay.mp4"
VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Clip      :", CLIP_PATH)
print("Recording :", RECORDING_PATH)
print("Selection :", SELECTION_PATH)
print("Asset     :", ASSET_PATH)

## 1. Replay & render the ventral view

The first run replays the clip (a few seconds) and renders a ventral-view video; subsequent runs reuse the cached recording. Delete `flybody_replay_recording.pkl` to force a fresh replay.

In [ ]:
from flygym import Simulation
from flygym.compose import ActuatorType, KinematicPosePreset, TetheredWorld
from flygym.compose.fly import FlyBody
from flygym.flybody import (
    FlyBodyActuatedDOFPreset,
    FlyBodyAxisOrder,
    FlyBodyJointPreset,
    FlyBodySkeleton,
)
from flygym.utils.math import Rotation3D

if RECORDING_PATH.exists():
    recording = load_replay_recording(RECORDING_PATH)
    print(
        f"Loaded cached recording ({recording.n_steps()} steps, dt={recording.timestep:.1e} s)"
    )
    fly = sim = None  # no render below
else:
    fly = FlyBody(name="flybody_template")
    skeleton = FlyBodySkeleton(
        axis_order=FlyBodyAxisOrder.YAW_ROLL_PITCH,
        joint_preset=FlyBodyJointPreset.LEGS_ONLY,
    )
    fly.add_joints(skeleton, KinematicPosePreset.FLYBODY_NEUTRAL)
    position_dofs = fly.skeleton.get_actuated_dofs_from_preset(
        FlyBodyActuatedDOFPreset.LEGS_ACTIVE_ONLY
    )
    fly.add_actuators(position_dofs, ActuatorType.POSITION, kp=1.0)
    fly.add_tendons()
    fly.add_tendon_actuators()

    ventral_cam = fly.add_tracking_camera(
        name="ventral",
        pos_offset=(0.0, 0.0, -4.0),
        rotation=Rotation3D("euler", (np.pi, 0.0, 0.0)),
        fovy=45.0,
    )
    world = TetheredWorld()
    world.add_fly(fly, (0, 0, 0), Rotation3D("quat", (1, 0, 0, 0)))

    sim = Simulation(world)

    renderer = sim.set_renderer(
        [ventral_cam],
        camera_res=(320, 320),
        playback_speed=0.1,
        output_fps=40,
    )

    recording, fly, sim = replay_clip(CLIP_PATH, fly=fly, sim=sim)
    save_replay_recording(recording, RECORDING_PATH)
    sim.renderer.save_video(VIDEO_PATH)
    print(f"Replay done ({recording.n_steps()} steps). Recording -> {RECORDING_PATH}")
    print(f"Ventral video -> {VIDEO_PATH}")

In [ ]:
# Play the ventral-view video inline. If the renderer was bypassed (cached
# recording) the cell falls back to the saved file.
from IPython.display import Video

if sim is not None:
    sim.renderer.show_in_notebook()
else:
    if VIDEO_PATH.exists():
        display(Video(str(VIDEO_PATH), embed=True))
    else:
        print(
            "No video on disk. Delete",
            RECORDING_PATH.name,
            "and rerun the replay cell to regenerate one.",
        )

## 2. Pick one step per leg position

The plot below shows each claw's **body-frame anteroposterior** coordinate (forward / back relative to the thorax) vs. time. Local maxima are anterior extreme positions (AEPs); they're overlaid as orange dots and printed below each plot. Phase 0 of a step cycle is conventionally the AEP, so a clean step is the slice from one AEP to the next.

Pick one step per leg position (front, middle, hind). Either side is fine; the opposite side is mirrored automatically when the asset is built.

In [ ]:
time_grid = np.arange(recording.n_steps()) * recording.timestep

candidates_per_leg = {}
fig, axes = plt.subplots(6, 1, figsize=(11, 9), sharex=True, tight_layout=True)
for leg_idx, leg in enumerate(LEGS):
    ap = recording.claw_body[:, leg_idx, 0]
    dv = recording.claw_body[:, leg_idx, 2]  # dorsoventral
    candidates = find_candidate_peps(ap, recording.timestep)
    candidates_per_leg[leg] = candidates
    axes[leg_idx].plot(time_grid, ap, lw=0.8)
    axes[leg_idx].plot(
        time_grid[candidates], ap[candidates], "o", ms=4, color="tab:orange"
    )
    twin_ax = axes[leg_idx].twinx()
    twin_ax.plot(time_grid, dv, lw=0.8, color="tab:green")
    twin_ax.set_ylabel("DV (mm)", color="tab:green")
    axes[leg_idx].set_ylabel(f"{leg.upper()}\n(mm)")
    axes[leg_idx].grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    "Body-frame anteroposterior claw position (forward = positive).  Orange dots = candidate AEPs."
)
plt.show()

print("AEP candidate sample indices per leg:")
for leg, candidates in candidates_per_leg.items():
    formatted = ", ".join(str(int(i)) for i in candidates)
    print(f"  {leg.upper()}: [{formatted}]")

### Type your picks into the cell below

For each of `F`, `M`, `H`:

- `side` — `"l"` or `"r"`, whichever side's plot above shows the cleanest cycle.
- `start` — sample index of the AEP that starts the cycle (the orange dot you choose).
- `end` — sample index of the **next** AEP on the same leg (start of the following cycle).

Skip the first ~0.15 s (initial ramp transient).

In [ ]:
selection = {
    "picks": {
        "F": {"side": "r", "start": 7736, "end": 9363},
        "M": {"side": "l", "start": 3421, "end": 5047},
        "H": {"side": "l", "start": 4558, "end": 6058},
    },
    "notes": (
        "Edit the start/end indices to match the AEP candidates printed above. "
        "Either side is fine; the other side is mirrored automatically."
    ),
}

save_selection(selection, SELECTION_PATH)
print(f"Wrote selection to {SELECTION_PATH}")

### Verify the picks visually

Each picked slice is replotted on top of the leg's full anteroposterior trace, with the cycle highlighted in orange. The blue-shaded regions inside the slice mark timesteps where the AP velocity is negative — i.e. the claw is retreating from AEP toward PEP. Their total fraction is the `swing_fraction` that will be written into the asset.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=False, tight_layout=True)
for ax, leg_pos in zip(axes, LEG_POSITIONS):
    entry = selection["picks"][leg_pos]
    leg = f"{entry['side']}{leg_pos.lower()}"
    start, end = int(entry["start"]), int(entry["end"])
    leg_idx = LEGS.index(leg)
    ap = recording.claw_body[:, leg_idx, 0]

    # Swing classification matches the asset builder:
    #   swing = timesteps where np.diff(claw_ap) < 0
    ap_slice = ap[start:end]
    ap_velocity = np.diff(ap_slice)
    swing_mask = ap_velocity > 0
    swing_frac = float(swing_mask.mean()) if swing_mask.size else 0.0

    lo = max(0, start - 200)
    hi = min(recording.n_steps(), end + 200)
    t = np.arange(lo, hi) * recording.timestep
    ax.plot(t, ap[lo:hi], lw=0.8, color="tab:blue", label="anteroposterior")
    ax.axvspan(
        start * recording.timestep,
        end * recording.timestep,
        color="orange",
        alpha=0.12,
        label="picked cycle",
    )

    # Per-sample swing shading inside the slice. `np.diff` returns N-1 values;
    # each entry covers the interval [i, i+1].
    swing_indices = np.where(swing_mask)[0]
    for i in swing_indices:
        ax.axvspan(
            (start + i) * recording.timestep,
            (start + i + 1) * recording.timestep,
            color="tab:blue",
            alpha=0.08,
            linewidth=0,
        )
    # Sentinel for the legend.
    ax.axvspan(np.nan, np.nan, color="tab:blue", alpha=0.08, label="swing (Δap>0)")

    ax.set_title(
        f"{leg_pos}: leg={leg.upper()}  cycle_len={end - start}  swing_fraction={swing_frac:.2f}"
    )
    ax.set_ylabel("AP position (mm)")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
plt.show()

## 3. Build the asset

Resample each pick onto the phase grid, mirror to the opposite side, and write `single_steps_flybody.pkl`.

In [ ]:
asset = build_asset_from_selection(recording, selection, n_phase_bins=200)
save_asset(asset, ASSET_PATH)
print(f"Wrote {ASSET_PATH}")
print()
print("Per-leg cycle lengths (samples) and swing fractions:")
for leg in LEGS:
    print(
        f"  {leg.upper()}: cycle_len={asset['meta']['cycle_lengths_samples'][leg]:4d}"
        f"  swing_fraction={asset['swing_fractions'][leg]:.2f}"
    )

## 4. Sanity-check the built cycles

Plot the seven DOFs of each leg over one phase cycle. Left and right of the same leg position should be roll/yaw-flipped (visible as inverted curves for DOFs labelled `roll` or `yaw`); pitch curves should overlap.

In [ ]:
from flygym_demo.complex_terrain.preprogrammed import _DOFS_PER_LEG

phase = np.linspace(0, 2 * np.pi, asset["meta"]["n_phase_bins"], endpoint=False)
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True, tight_layout=True)
for ax, leg_pos in zip(axes, LEG_POSITIONS):
    left = asset["joint_angles"][f"l{leg_pos.lower()}"]
    right = asset["joint_angles"][f"r{leg_pos.lower()}"]
    for dof_idx, (parent, child, axis) in enumerate(_DOFS_PER_LEG):
        ax.plot(phase, left[dof_idx], lw=1.0, label=f"L {child}-{axis}")
        ax.plot(phase, right[dof_idx], lw=1.0, ls="--", label=f"R {child}-{axis}")
    swing_end = asset["swing_fractions"][f"l{leg_pos.lower()}"] * 2 * np.pi
    ax.axvspan(0, swing_end, color="orange", alpha=0.1, label="swing")
    ax.set_title(f"Leg position {leg_pos}")
    ax.set_ylabel("angle (rad)")
    ax.grid(True, alpha=0.3)
    ax.legend(ncols=4, fontsize=6, loc="upper right")
axes[-1].set_xlabel("Phase (rad)")
plt.show()